# Mission 3: KLUE-RoBERTa Pure-Nausea Sampling

KLUE-RoBERTa plain BCE, seed 42, truncate baseline에 pure-nausea group-aware sampling만 추가하는 실험 노트북이다. 학습 구현은 `train.py`와 `m3` 모듈을 재사용한다.

## 0. 환경 및 모듈 준비

현재 실행 위치에서 `mission3_symptom` 디렉터리를 찾고 기존 평가 모듈을 불러온다.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

mission3_dir = Path.cwd()
if not (mission3_dir / "m3").is_dir():
    mission3_dir = mission3_dir / "mission3_symptom"
if not (mission3_dir / "m3").is_dir():
    raise RuntimeError("mission3_symptom 디렉터리에서 실행하거나 repository root에서 실행하세요.")
sys.path.insert(0, str(mission3_dir))

from m3 import TARGET_SYMPTOMS, apply_thresholds, eval_macro_f1

plt.rcParams["axes.unicode_minus"] = False
print(f"Mission 3 directory: {mission3_dir.resolve()}")
print(f"Python executable: {Path(sys.executable).resolve()}")

def run_command_live(command):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("학습 프로세스의 출력을 연결하지 못했습니다.")

    output_buffer = []
    while True:
        character = process.stdout.read(1)
        if character == "":
            if output_buffer:
                print("".join(output_buffer), end="", flush=True)
            break
        output_buffer.append(character)
        if character in {"\r", "\n"}:
            print("".join(output_buffer), end="", flush=True)
            output_buffer.clear()

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## 1. 학습 설정

실험 전에 주로 변경하는 값을 한곳에서 관리한다. CSV는 기본적으로 `mission3_symptom/data` 아래에 있다고 가정하며, 다른 위치에 있다면 `TRAIN_CSV`와 `VAL_CSV`만 수정한다.

In [ ]:
TRAIN_CSV = mission3_dir.parent / "data" / "csv" / "mission3_train.csv"
VAL_CSV = mission3_dir.parent / "data" / "csv" / "mission3_val.csv"
RUN_NAME = "kf_deberta_base_plain_bce_seed42_val_loss"
OUTPUT_DIR = mission3_dir / "runs" / RUN_NAME
PREFLIGHT_DIR = mission3_dir / "runs" / f"{RUN_NAME}_preflight"
PREFLIGHT_REPORT = PREFLIGHT_DIR / "preflight_report.json"

MODEL_NAME = "kakaobank/kf-deberta-base"
MODEL_REVISION = "363b171d71443b0874b0bf9cea053eb5b1650633"
SEED = 42
MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 8
VAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EPOCHS = 3
USE_AMP = True
USE_POS_WEIGHT = False
LOSS_TYPE = "bce"
PAIRWISE_ALPHA = 0.1
ENCODE_MODE = "truncate"
POOLING_TYPE = "cls"
CHECKPOINT_METRIC = "val_loss"
USE_PURE_NAUSEA_SAMPLING = False
PURE_NAUSEA_WEIGHT = 1.5
ASL_GAMMA_NEG = 4.0
ASL_GAMMA_POS = 1.0
ASL_CLIP = 0.05
ASL_EPS = 1e-8
ASL_REDUCTION = "mean"

missing_csv = [path for path in (TRAIN_CSV, VAL_CSV) if not path.is_file()]
if missing_csv:
    missing_text = "\n".join(f"- {path.resolve()}" for path in missing_csv)
    raise FileNotFoundError(
        "CSV 파일을 찾을 수 없습니다. TRAIN_CSV와 VAL_CSV를 확인하세요.\n"
        + missing_text
    )

settings = {
    "TRAIN_CSV": TRAIN_CSV.resolve(),
    "VAL_CSV": VAL_CSV.resolve(),
    "RUN_NAME": RUN_NAME,
    "OUTPUT_DIR": OUTPUT_DIR.resolve(),
    "MODEL_NAME": MODEL_NAME,
    "MODEL_REVISION": MODEL_REVISION,
    "SEED": SEED,
    "MAX_LENGTH": MAX_LENGTH,
    "TRAIN_BATCH_SIZE": TRAIN_BATCH_SIZE,
    "VAL_BATCH_SIZE": VAL_BATCH_SIZE,
    "GRAD_ACCUM_STEPS": GRAD_ACCUM_STEPS,
    "LEARNING_RATE": LEARNING_RATE,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "WARMUP_RATIO": WARMUP_RATIO,
    "EPOCHS": EPOCHS,
    "USE_AMP": USE_AMP,
    "USE_POS_WEIGHT": USE_POS_WEIGHT,
    "LOSS_TYPE": LOSS_TYPE,
    "PAIRWISE_ALPHA": PAIRWISE_ALPHA,
    "ENCODE_MODE": ENCODE_MODE,
    "POOLING_TYPE": POOLING_TYPE,
    "CHECKPOINT_METRIC": CHECKPOINT_METRIC,
    "USE_PURE_NAUSEA_SAMPLING": USE_PURE_NAUSEA_SAMPLING,
    "PURE_NAUSEA_WEIGHT": PURE_NAUSEA_WEIGHT,
    "ASL_GAMMA_NEG": ASL_GAMMA_NEG,
    "ASL_GAMMA_POS": ASL_GAMMA_POS,
    "ASL_CLIP": ASL_CLIP,
    "ASL_EPS": ASL_EPS,
    "ASL_REDUCTION": ASL_REDUCTION,
}
for name, value in settings.items():
    print(f"{name}: {value}")

## 2. KF-DeBERTa Compatibility and Memory Preflight

이 셀은 성능 평가나 Training loop가 아니다. Pinned KF-DeBERTa의 tokenizer/config/model 계약, aggregate UNK·길이 통계, batch 8 × sequence 512 AMP forward/backward와 **optimizer step 1회**, validation batch 16 forward 1회, offline save/load를 확인한다.

Training/Validation CSV는 tokenizer 통계에만 사용하며 모델 update에는 사용하지 않는다. 출력에는 원문이나 call_id가 포함되지 않으며 `runs/<RUN_NAME>_preflight`는 Git에서 제외된다.

`full_training_train_batch=8`, `full_training_val_batch=16`, offline round-trip success를 확인한 뒤에만 Full Training 셀을 실행한다. OOM이면 스크립트는 train batch 4를 한 번 시도하고 gradient accumulation 4를 보고하며, 그 외 조건은 자동 변경하지 않는다.

In [ ]:
preflight_command = [
    sys.executable,
    "-u",
    str(mission3_dir / "kf_deberta_preflight.py"),
    "--train-csv", str(TRAIN_CSV),
    "--val-csv", str(VAL_CSV),
    "--model-name-or-path", MODEL_NAME,
    "--model-revision", MODEL_REVISION,
    "--output-json", str(PREFLIGHT_REPORT),
]

print("KF-DeBERTa preflight 시작")
run_command_live(preflight_command)
with open(PREFLIGHT_REPORT, encoding="utf-8") as file:
    preflight_report = json.load(file)
print(json.dumps({
    "contract": preflight_report["contract"],
    "tokenizer_statistics": preflight_report["tokenizer_statistics"],
    "train_memory": preflight_report["train_memory"],
    "validation_memory": preflight_report["validation_memory"],
    "offline_round_trip": preflight_report["offline_round_trip"],
}, ensure_ascii=False, indent=2))

## 3. Full Training

이 단계는 Train 전체 데이터와 Validation 전체 데이터를 사용하는 정식 학습이다. 1번 셀에서 선택한 loss를 사용하고 validation loss가 가장 낮은 checkpoint를 저장한다. Threshold 0.5 Macro F1과 클래스별 F1은 기존처럼 모두 기록한다.

학습 loop를 노트북에 중복 구현하지 않고 1번 셀의 설정으로 기존 `train.py`를 실행한다. `OUTPUT_DIR`이 비어 있지 않으면 기존 정식 run을 보호하기 위해 실행이 중단되므로, 새로운 실험은 `RUN_NAME`을 변경한다.

In [ ]:
full_train_command = [
    sys.executable,
    "-u",
    str(mission3_dir / "train.py"),
    "--train-csv", str(TRAIN_CSV),
    "--val-csv", str(VAL_CSV),
    "--output-dir", str(OUTPUT_DIR),
    "--model-name-or-path", MODEL_NAME,
    "--model-revision", MODEL_REVISION,
    "--seed", str(SEED),
    "--max-length", str(MAX_LENGTH),
    "--train-batch-size", str(TRAIN_BATCH_SIZE),
    "--val-batch-size", str(VAL_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
    "--learning-rate", str(LEARNING_RATE),
    "--weight-decay", str(WEIGHT_DECAY),
    "--warmup-ratio", str(WARMUP_RATIO),
    "--epochs", str(EPOCHS),
    "--loss-type", LOSS_TYPE,
    "--pairwise-alpha", str(PAIRWISE_ALPHA),
    "--encode-mode", ENCODE_MODE,
    "--pooling-type", POOLING_TYPE,
    "--checkpoint-metric", CHECKPOINT_METRIC,
    "--pure-nausea-weight", str(PURE_NAUSEA_WEIGHT),
    "--asl-gamma-neg", str(ASL_GAMMA_NEG),
    "--asl-gamma-pos", str(ASL_GAMMA_POS),
    "--asl-clip", str(ASL_CLIP),
    "--asl-eps", str(ASL_EPS),
    "--asl-reduction", ASL_REDUCTION,
]
if USE_AMP:
    full_train_command.append("--amp")
if USE_POS_WEIGHT:
    full_train_command.append("--use-pos-weight")
if USE_PURE_NAUSEA_SAMPLING:
    full_train_command.append("--use-pure-nausea-sampling")

print("Full Training 시작")
run_command_live(full_train_command)
print(f"Full Training 완료: {OUTPUT_DIR}")

## 4. 정식 Run 산출물 로드

- `run_config.json`: 실제 학습 설정, 데이터 규모, parameter 수, 실행 환경, 토큰 길이 통계
- `history.json`: epoch별 train/validation loss와 threshold 0.5 성능
- `baseline_metrics.json`: best checkpoint의 최종 Validation 결과와 추론 시간
- `val_logits.npy`: best checkpoint의 sigmoid 적용 전 출력
- `val_probs.npy`: logits에 sigmoid를 적용한 확률
- `val_labels.npy`: 동일 순서의 실제 9개 Validation label

In [ ]:
run_dir = OUTPUT_DIR
required_files = [
    "run_config.json",
    "history.json",
    "baseline_metrics.json",
    "val_logits.npy",
    "val_probs.npy",
    "val_labels.npy",
]
missing = [name for name in required_files if not (run_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"정식 run 산출물이 없습니다: {missing}. 먼저 Full Training을 완료하세요."
    )

with open(run_dir / "run_config.json", encoding="utf-8") as file:
    run_config = json.load(file)
with open(run_dir / "history.json", encoding="utf-8") as file:
    history = json.load(file)["epochs"]
with open(run_dir / "baseline_metrics.json", encoding="utf-8") as file:
    saved_metrics = json.load(file)

val_logits = np.load(run_dir / "val_logits.npy")
val_probs = np.load(run_dir / "val_probs.npy")
val_labels = np.load(run_dir / "val_labels.npy")

print(f"Run directory: {run_dir.resolve()}")
print(f"Best epoch: {saved_metrics['best_epoch']}")
print(f"Validation shape: {val_probs.shape}")
print(json.dumps(run_config, ensure_ascii=False, indent=2))

## 5. Threshold 0.5 결과 재검증

저장된 실제 Validation probability에 기존 `apply_thresholds()`와 `eval_macro_f1()`을 다시 적용하여 저장 결과와 일치하는지 확인한다.

In [ ]:
if val_labels.ndim != 2 or val_labels.shape[1] != len(TARGET_SYMPTOMS):
    raise ValueError(f"Validation label 형상이 올바르지 않습니다: {val_labels.shape}")
if val_logits.shape != val_labels.shape or val_probs.shape != val_labels.shape:
    raise ValueError("Validation logits/probabilities/labels 형상이 일치하지 않습니다.")

predictions = apply_thresholds(val_probs, 0.5)
macro_f1, per_class_f1 = eval_macro_f1(
    val_labels, predictions, return_per_class=True
)
if not np.isclose(macro_f1, saved_metrics["val_macro_f1"]):
    raise ValueError("저장된 baseline metric과 재계산 결과가 다릅니다.")

print(f"Macro F1 @ 0.5: {macro_f1:.4f}")
pd.DataFrame({
    "symptom": TARGET_SYMPTOMS,
    "f1": [per_class_f1[symptom] for symptom in TARGET_SYMPTOMS],
})

## 6. 학습 이력 시각화

`history.json`에 저장된 epoch별 loss와 threshold 0.5 Validation Macro F1을 확인한다.

In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(history_df["epoch"], history_df["val_macro_f1_at_0_5"], marker="o")
axes[1].set_title("Validation Macro F1 @ 0.5")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 결정 임계값 — 0.5 고정 (대회 규정)

대회 공지(2026-09-25, Decision Threshold FAQ Q1~Q5)에 따라 최종 확률을 0/1 로 바꾸는 임계값은 **9개 클래스 모두 0.5 고정**이다. Train 분할·OOF·Validation 어디서 고른 값이든 0.5 가 아닌 임계값과 클래스별 임계값은 threshold tuning 이라 쓰지 않는다. 그래서 예전의 클래스별 임계값 탐색·적용 절(7·8절)은 이 노트북에서 제거했다.

0.5 에서 소수 클래스 recall 이 낮은 문제는 임계값이 아니라 학습 손실로 다룬다 (공지 Q4 의 calibration 을 고려한 loss 설계): `--use-pos-weight --pos-weight-power 0.5`. 제출 경로 `m3/infer.py` 도 0.5 하나만 쓴다 (`DECISION_THRESHOLD`).